# Focusing grating-coupler PCell: chip to fiber

This reciprocal setup excites the parameterized GDSFactory cell through its waveguide port and projects the radiated field onto the time-reversed Gaussian fiber mode. It provides an independent check of the [fiber-to-chip example](fdtd_focusing_grating_coupler.ipynb), which follows [FDTD-Bench problem 07](https://github.com/doplaydo/FDTD-Bench/tree/main/problems/07_focusing_grating_coupler) and the [Tidy3D FocusedApodGC design](https://www.flexcompute.com/tidy3d/examples/notebooks/FocusedApodGC/).

In [ ]:
from pathlib import Path
from runpy import run_path

import gdsfactory as gf

from gsim import fdtd

PCELL_PATH = Path("gds/build_focusing_gc.py")
if not PCELL_PATH.is_file():
    PCELL_PATH = Path("nbs") / PCELL_PATH

STRAIGHT_WAVEGUIDE_LENGTH_UM = 2.0
FIBER_CENTER_UM = (22.4041, 0.0, 1.05)
FIBER_SIZE_UM = (12.48, 12.48, 0.0)
FIBER_DIRECTION = (0.17388, 0.0, 0.98477)
DEVICE_CENTER_X_UM = 17.136
DEVICE_SPAN_X_UM = 34.272

## Build the same generic-PDK component

In [ ]:
pdk = gf.gpdk.PDK
pdk.activate()
focusing_grating_coupler = run_path(PCELL_PATH)["focusing_grating_coupler"]
component = focusing_grating_coupler(waveguide_length_um=STRAIGHT_WAVEGUIDE_LENGTH_UM)
component

## Excite the waveguide and monitor the fiber mode

The receiver uses the opposite propagation vector from the in-coupling beam. Under an eigenmode source, ZapFDTD normalizes this overlap like an S-parameter.

In [ ]:
simulation = fdtd.Simulation(pdk=pdk)
simulation.materials(background="sio2", overrides={"si": 3.47, "sio2": 1.44})
simulation.geometry(component, mesh_size_nm=500, geometry_tolerance_nm=20)
simulation.source(
    port="o1",
    wavelength_um=1.55,
    wavelength_span_um=0.1,
    num_wavelengths=101,
)
simulation.monitors.add_plane(
    name="fiber",
    center_um=FIBER_CENTER_UM,
    size_um=FIBER_SIZE_UM,
    normal="+z",
    flux=True,
    heatmap=fdtd.Heatmap(quantity="abs_e", wavelengths_um=[1.55]),
    fiber_mode=fdtd.FiberMode(
        propagation_direction=FIBER_DIRECTION,
        e_polarization=(0.0, 1.0, 0.0),
        focal_point_um=FIBER_CENTER_UM,
        waist_radius_um=5.2,
        refractive_index=1.44,
    ),
)
simulation.monitors.add_plane(
    name="cross",
    center_um=(DEVICE_CENTER_X_UM, 0.0, 0.0),
    size_um=(DEVICE_SPAN_X_UM, 0.0, 4.0),
    normal="+y",
    flux=False,
    heatmap=fdtd.Heatmap(quantity="abs_e", wavelengths_um=[1.55]),
)
simulation.domain(padding_um=0.5, pml_cells=32)
simulation.solver(cell_size_nm=50, energy_decay_fraction=1e-5)

In [ ]:
simulation.plot_3d()

In [ ]:
result = simulation.run(check_cache=True)

## Coupling and fields

The solver returns a complex normalized overlap coefficient. Its magnitude squared is the chip-to-fiber power efficiency.

In [ ]:
coupling = fdtd.fiber_coupling_efficiency(result.monitors["fiber"])
coupling.plot_plotly()

In [ ]:
result.monitors["fiber"].plot_heatmap(
    wavelength_um=1.55, quantity="abs_e", cmap="magma"
)

In [ ]:
result.monitors["cross"].plot_heatmap(
    wavelength_um=1.55, quantity="abs_e", cmap="magma"
)

For a reciprocal linear structure, this spectrum should approach the fiber-to-chip result when both runs use the same geometry, grid, aperture, and mode definitions. Residual differences are a useful convergence diagnostic.